##  TechMind — Exploración y Preparación del Dataset **OpenAlex**

#### Equipo tejONEs

#### 03_exploracion_dataset_openalex.ipynb

💡**Dataset**: [Open Alex - Extracción a traves de la API Oficial](https://api.openalex.org/works)

- El proceso general que sigue este pipeline es:
    - Extracción por términos técnicos desde la API y persistencia del payload crudo en JSON y CSV
    - Reconstrucción de resúmenes y normalización al esquema `titulo`, `texto`, `categoria`, `autor`, `tipo`
    - Mapeo a las siete categorías mediante título, resumen y conceptos de OpenAlex
    - Limpieza de texto (HTML, URLs, nulos, duplicados y filtro de mínimo 100 palabras)
    - Refuerzo de categorías con búsquedas adicionales y selección sin reemplazo según cuotas complementarias
    - Traducción al español de título y texto, procesamiento NLP y respaldo de traducciones
    - Validación de cuotas e integridad, y exportación del dataset final en `procesados/`

# Pipeline de Datos (OpenAlex)

### Adquisición, limpieza, estructuración y balanceo de datos

**Fuente:** API pública de OpenAlex (`https://api.openalex.org/works`)

Este notebook ejecuta de punta a punta:
- **Fase 1:** Ingesta cruda completa (sin descartar columnas del payload original).
- **Fase 2:** Transformación y normalización al esquema final de 5 columnas.
- **Fase 3:** Segmentación en los 7 temas técnicos y selección según cuotas complementarias.

El dataset resultante alimentará un modelo de NLP para clasificación automática y búsqueda semántica (TF-IDF / Embeddings), por lo que la columna `texto` debe quedar libre de nulos y de etiquetas HTML.

## Importaciones

In [ ]:
import os
from pathlib import Path

import nltk
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import requests #Para conectarse a Internet, hacer peticiones a servidores y descargar la información de las APIs.
import pandas as pd
import numpy as np
import json #Para trabajar con el formato JSON que la API devuelve
import re #para buscar, extraer y limpiar patrones complejos de texto
import time
from collections import Counter # Para contar fácilmente las palabras más frecuentes en los títulos o qué autores publican más.

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))

## Configuración

In [ ]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / "data_science").exists() and (candidate / "README.md").exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / "data_science" / "data").resolve())
CARPETA_CRUDOS = str((project_root / "data_science" / "data" / "crudos").resolve())
CARPETA_PROCESADOS = str((project_root / "data_science" / "data" / "procesados").resolve())


print(f'📁 Carpeta proyecto local: {project_root.name}')

for nombre, ruta in [
    ("datos", CARPETA_DATA),
    ("datos crudos", CARPETA_CRUDOS),
    ("datos procesados", CARPETA_PROCESADOS),
]:
    path = Path(ruta)
    if path.exists():
        print(f"✅📂 Carpeta {nombre}: {path.relative_to(project_root)}")
    else:
        print(f"❌ No se encontro la carpeta de {nombre}❌")

####  Buenas prácticas con la API de OpenAlex
OpenAlex es gratuita y no requiere API key, pero recomienda identificarse mediante el parámetro `mailto` para acceder al *polite pool* (menor latencia, menos throttling). Así que coloqué mi email

In [ ]:
# Identificación para el 'polite pool' de OpenAlex (recomendado, no obligatorio)
MAILTO = os.getenv('OPENALEX_MAILTO')  # Contacto opcional para acceder al polite pool
BASE_URL = 'https://api.openalex.org/works'

OpenAlex está organizado por recursos. El dominio https://api.openalex.org es solo la entrada al servicio; los datos reales están en endpoints como:

- /works → publicaciones científicas.
- /authors → autores.
- /institutions → instituciones.
- /sources → revistas y conferencias.
- /topics → temas.

# Fase 1: Ingesta Cruda de Artículos Científicos desde OpenAlex

## Objetivo

Esta etapa inicia el proceso de recolección de información científica consultando la API pública de OpenAlex. El propósito es obtener un conjunto amplio de artículos relacionados con las siete áreas tecnológicas definidas previamente (Backend, Frontend, Data Science, Cloud, DevOps, Bases de Datos y Mobile), conservando inicialmente toda la información proporcionada por la API sin realizar ningún tipo de limpieza o transformación.

## ¿Cómo funciona?

El proceso recorre secuencialmente cada categoría técnica definida en `SEARCH_TERMS`. Para cada una de ellas se invoca la función `fetch_category_raw()`, la cual:

- Consulta OpenAlex utilizando múltiples términos de búsqueda asociados a la categoría.
- Recupera varias páginas de resultados de la API.
- Elimina artículos duplicados utilizando el identificador único (`id`) de OpenAlex.
- Conserva el payload completo devuelto por la API.
- Añade únicamente dos campos de trazabilidad (`_search_category` y `_search_term`) para registrar desde qué categoría y término fue encontrado cada artículo.

Una vez finalizada la consulta de una categoría, sus registros se agregan a una colección global denominada `raw_records`.

## Resultado esperado

Al finalizar esta fase se dispone de una única colección (`raw_records`) que contiene todos los artículos científicos recuperados desde OpenAlex para las siete áreas de conocimiento.

Estos datos aún son considerados **datos crudos (raw data)**, ya que pueden contener información innecesaria, formatos inconsistentes, valores faltantes o campos que posteriormente serán normalizados durante la etapa de limpieza y preprocesamiento.

In [ ]:
# Términos de búsqueda por tema técnico (múltiples términos por tema para asegurar cobertura y recall)
SEARCH_TERMS = {
    'Backend': [
        'backend development', 'server-side programming', 'REST API design',
        'microservices architecture', 'API gateway design'
    ],
    'Frontend': [
        'frontend web development', 'JavaScript framework', 'responsive web design',
        'single page application', 'user interface development'
    ],
    'Data Science': [
        'data science', 'machine learning', 'data analytics',
        'artificial intelligence applications', 'data mining techniques'
    ],
    'Cloud': [
        'cloud computing', 'cloud infrastructure', 'serverless computing',
        'cloud native architecture', 'edge cloud computing'
    ],
    'DevOps': [
        'DevOps practices', 'continuous integration continuous deployment',
        'container orchestration kubernetes', 'infrastructure as code',
        'software delivery automation'
    ],
    'Bases de Datos': [
        'database management systems', 'SQL database design', 'NoSQL database',
        'distributed database systems', 'data warehouse architecture'
    ],
    'Mobile': [
        'mobile application development', 'Android application development',
        'iOS application development', 'cross platform mobile development',
        'mobile computing'
    ],
}

# Cantidad mínima de registros crudos (antes de limpieza) que intentamos reunir por tema.
# Se pide de más porque limpieza + deduplicado siempre reduce el volumen.
RAW_TARGET_PER_CATEGORY = 50

# Cuotas que complementan los conteos de Coursera, Microsoft Learn y StackExchange
TARGET_COUNTS_OPENALEX = {
    'Backend': 55,
    'Frontend': 79,
    'Data Science': 50,
    'Cloud': 50,
    'DevOps': 83,
    'Bases de Datos': 50,
    'Mobile': 96,
}
TRANSLATION_BUFFER_OPENALEX = 15
CANDIDATE_TARGET_COUNTS_OPENALEX = {
    category: target + TRANSLATION_BUFFER_OPENALEX
    for category, target in TARGET_COUNTS_OPENALEX.items()
}
PER_PAGE = 70  # máximo permitido por OpenAlex por página

El script utiliza dos funciones organizadas jerárquicamente: la primera se encarga puramente de hablar con Internet, y la segunda se encarga de la lógica de negocio (iterar, filtrar y acumular los datos).

In [ ]:
#Función 1
def fetch_openalex_page(query, page=1, per_page=PER_PAGE, mailto=MAILTO):
    """
    Realiza una única petición GET a la API de OpenAlex y devuelve la respuesta JSON completa
    (sin transformar), incluyendo 'meta' y la lista 'results' con el payload íntegro de cada obra.
    """
    params = {
        'search': query,
        'per-page': per_page,
        'page': page,
        'mailto': mailto,
    }
    #Aprovechamos que la librería requests nos permite pasarle un diccionario de Python a través del parámetro params=
    response = requests.get(BASE_URL, params=params, timeout=30)
    #Realizamos la petición GET a la API. Si este no responde en 30 segundos, cancela la petición y lanza un error
    response.raise_for_status()
    # Verificador de estado HTTP: Lanza un error automático si el servidor responde con un fallo
    return response.json()
    # Método .json() de requests toma ese texto plano y usa la lógica del módulo json para transformarlo automáticamente en un diccionario/lista nativo de Python.


Hice una única consulta ala api para verificar y me dió decenas de campos como: Título, Año de publicación, DOI, Resumen (si está disponible), etc

In [ ]:
#Hacer una única consulta
#respuesta = fetch_openalex_page("data science")
#
# Mostrar todo el JSON de forma legible
#from pprint import pprint
#pprint(respuesta)

En esta segunda función no se hace una única consulta, sino que automatiza muchas consultas recorriendo la lista de términos de búsqueda (el diccionario SEARCH_TERMS). Aquí hay un recorrido externo sobre los términos y otro interno sobre los artículos devueltos por cada búsqueda.

In [ ]:
#Función 2
def fetch_category_raw(category, terms, raw_target=RAW_TARGET_PER_CATEGORY, max_pages_per_term=2):
    """
    Recorre la lista de términos de búsqueda de un tema hasta reunir al menos `raw_target`
    registros crudos únicos (deduplicados por 'id' de OpenAlex). Cada registro conserva
    TODO el payload original de la API, y se le añade metadata propia de trazabilidad
    ('_search_category' y '_search_term') SIN eliminar ningún campo original.
    """
    collected = {}
    #Diccionario vacío (no una lista). Para eliminar duplicados más adelante.
    printed_raw = False  # Permite mostrar únicamente el primer JSON crudo de OpenAlex.
    for term in terms:
        if len(collected) >= raw_target:
            break
        #Si en cualquier momento len(collected) llega o supera la meta, el bucle se detiene.
        page = 1
        """
        Bucle while que avanza página por página (page=1, page=2...) mientras no supere el límite
        de páginas permitido (max_pages_per_term) y aún falten artículos para la meta.
        """
        while page <= max_pages_per_term and len(collected) < raw_target:
            try:
                data = fetch_openalex_page(term, page=page)
            except requests.exceptions.RequestException as e:
                #as e: crea una variable llamada e que almacena el mensaje detallado del error generado por el sistema.
                print(f'  Error consultando "{term}" (página {page}): {e}')
                break

            results = data.get('results', [])
            if not results:
                break
                """
                Manejo de Errores: Si la función fetch_openalex_page falla, el bloque except atrapa la falla,
                imprime el problema en pantalla y rompe el bucle actual.
                """

            for work in results:
                work_id = work.get('id')
                #work representa cada elemento individual de la lista results.
                if work_id and work_id not in collected:
                    # Se preserva el payload crudo completo tal cual llega de la API
                    enriched = dict(work)
                    #Crea un nuevo diccionario independiente para evitar modificar el objeto original por referencia
                    enriched['_search_category'] = category
                    enriched['_search_term'] = term
                    collected[work_id] = enriched
                    """
                    - Deduplicación rápida mediante el id de OpenAlex.
                    - _search_category y _search_term son metadatos agregados por el programa.
                    """
            page += 1 #Incrementa el contador para pedir la siguiente página.
            time.sleep(0.15)  # respeto de rate-limit / buena práctica ante la API pública

    return list(collected.values())
    #Extrae únicamente los artículos (los valores del diccionario collected) y los devuelve como una lista limpia de diccionarios.

aquí probe la segunda función pasandole del dict "SEARCH_TERMS" la categoría de backend y un único elemento de la lista de terminos que es el segundo parámetro que recibe la función

In [ ]:
#backend_data = fetch_category_raw(
#    "Backend",
#    SEARCH_TERMS["Backend"],
#    raw_target=1
#)

Luego aquí se ejecuta la fase de ingesta cruda para todas las categorías definidas

In [ ]:
# Lista donde se almacenarán todos los artículos obtenidos desde OpenAlex.
# Cada elemento corresponde a un artículo científico representado como un diccionario de Python.
raw_records = []
print('=== FASE 1: Ingesta cruda desde OpenAlex ===\n')
# Recorre cada categoría técnica y su lista de términos de búsqueda.
for category, terms in SEARCH_TERMS.items():
    print(f'Consultando tema: {category}...')
    # Invoca la función encargada de consultar OpenAlex para la categoría actual.
    # La función devuelve una lista de artículos científicos sin procesar.
    candidate_target = CANDIDATE_TARGET_COUNTS_OPENALEX[category]
    raw_target = max(candidate_target * 3, 150)
    category_records = fetch_category_raw(
        category, terms, raw_target=raw_target, max_pages_per_term=4
    )
    # Muestra la cantidad de artículos únicos recuperados para la categoría.
    print(f'  -> {len(category_records)} registros crudos únicos obtenidos.')
    # Agrega los registros recuperados a la colección global.
    raw_records.extend(category_records)
# Muestra la cantidad total de artículos obtenidos considerando todas las categorías.
print(f'\nTotal de registros crudos (todas las categorías): {len(raw_records)}')
print("Proceso terminado")

#### Persistencia del crudo
Guardamos el payload íntegro en dos formatos dentro de `/Datos_crudos`:
- `openalex_raw.json`: estructura anidada tal cual la entrega la API (sin pérdida de metadatos).
- `openalex_raw.csv`: misma información en tabla plana, serializando en JSON los campos anidados (`authorships`, `concepts`, `abstract_inverted_index`, etc.) para que ninguna columna se pierda.

In [ ]:
#Guardado del crudo en formato JSON (preserva la estructura anidada original al 100%)
raw_json_path = f'{CARPETA_CRUDOS}/openalex_raw.json'
with open(raw_json_path, 'w', encoding='utf-8') as f:
    json.dump(raw_records, f, ensure_ascii=False)
print(f'Crudo JSON guardado: {Path(raw_json_path).relative_to(project_root)}')

# Guardado del crudo en formato CSV (columnas anidadas serializadas como texto JSON, sin descartarlas)
df_raw = pd.json_normalize(raw_records)
for col in df_raw.columns:
    if df_raw[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_raw[col] = df_raw[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)

raw_csv_path = f'{CARPETA_CRUDOS}/openalex_raw.csv'
df_raw.to_csv(raw_csv_path, index=False)
print(f'Crudo CSV guardado: {Path(raw_csv_path).relative_to(project_root)}')
print(f'Columnas capturadas del payload original: {len(df_raw.columns)}')
df_raw.sample(min(3, len(df_raw)))

### Encabezados del dataset crudo

In [ ]:
print(f'=== ENCABEZADOS DEL DATASET OPENALEX ({len(df_raw.columns)}) ===')
for numero, encabezado in enumerate(df_raw.columns, start=1):
    print(f'{numero}. {encabezado}')

## FASE 2: Transformación, Normalización y Construcción de Esquema

A partir de `raw_records` construimos el esquema final de 5 columnas:
`titulo`, `texto`, `categoria`, `autor`, `tipo`.

In [ ]:
def reconstruct_abstract(inverted_index):
    """
    OpenAlex entrega el abstract como un índice invertido {palabra: [posiciones]}
    por razones de licenciamiento. Aquí se reconstruye el texto plano en orden original.
    """
    if not inverted_index or not isinstance(inverted_index, dict):
      #Verifica que realmente recibió un diccionario.
        return ''
    max_pos = max(pos for positions in inverted_index.values() for pos in positions)
    #Aquí busca la posición más grande.
    slots = [''] * (max_pos + 1)
    #Crea una lista vacía con espacio suficiente.
    for word, positions in inverted_index.items():
    #Recorre cada palabra.
        for pos in positions:
          #Algunas palabras aparecen varias veces.
            slots[pos] = word
            #Va colocando cada palabra en su posición.
    return ' '.join(slots).strip()
    #Une todas las palabras.


def get_first_author(authorships):
  #Recibe la lista de autores y devuelve únicamente el primero o 'OpenAlex Contributor' si no existe.
    try:
        if authorships:
            name = authorships[0].get('author', {}).get('display_name')
            if name:
                return name
    except (AttributeError, IndexError, TypeError):
        pass
    return 'OpenAlex Contributor'
    #Si ocurre cualquier problema (lista vacía, dato mal formado, etc.), devuelve:

#OpenAlex también devuelve una lista de conceptos.
def get_concepts_keywords(concepts, top_n=6):
    """Extrae los N conceptos de mayor score de OpenAlex como texto de temario/keywords."""
    try:
        ordered = sorted(concepts or [], key=lambda c: c.get('score', 0), reverse=True)
        names = [c.get('display_name', '') for c in ordered[:top_n] if c.get('display_name')]
        return ', '.join(names)
    except (AttributeError, TypeError):
        return ''

#Este diccionario convierte distintos nombres de OpenAlex en un conjunto pequeño de categorías.
TIPO_MAPPING = {
    'article': 'articulo',
    'journal-article': 'articulo',
    'preprint': 'articulo',
    'proceedings-article': 'articulo',
    'dissertation': 'tesis',
    'thesis': 'tesis',
    'book-chapter': 'capitulo',
    'chapter': 'capitulo',
    'report': 'reporte',
    'monograph': 'reporte',
    'book': 'capitulo',
}
#Esto facilita el análisis posterior porque no tienes veinte tipos distintos.

#Hace una búsqueda en ese diccionario.
def normalize_tipo(openalex_type):
    """Normaliza el campo 'type' de OpenAlex a: articulo, tesis, capitulo o reporte."""
    return TIPO_MAPPING.get(str(openalex_type).lower(), 'reporte')

In [ ]:
# Diccionario de palabras clave (bilingüe) para el mapeo semántico a los 7 temas técnicos.
# Se aplica sobre título + abstract + conceptos de cada obra.
CATEGORY_KEYWORDS = {
    'Backend': ['backend', 'server-side', 'api', 'microservice', 'rest api', 'node.js', 'spring boot', '.net', 'java ee'],
    'Frontend': ['frontend', 'front-end', 'react', 'angular', 'vue', 'javascript', 'css', 'user interface', 'single page application'],
    'Data Science': ['data science', 'machine learning', 'deep learning', 'artificial intelligence', 'data analytics', 'neural network', 'data mining', 'big data'],
    'Cloud': ['cloud computing', 'cloud infrastructure', 'aws', 'azure', 'google cloud', 'serverless', 'cloud native', 'saas', 'paas', 'iaas'],
    'DevOps': ['devops', 'continuous integration', 'continuous deployment', 'ci/cd', 'kubernetes', 'docker', 'container orchestration', 'infrastructure as code'],
    'Bases de Datos': ['database', 'sql', 'nosql', 'relational database', 'data warehouse', 'postgresql', 'mongodb', 'database management'],
    'Mobile': ['mobile application', 'android', 'ios', 'mobile computing', 'flutter', 'react native', 'mobile app development', 'smartphone'],
}


def assign_category(titulo, texto, concepts_keywords, fallback_category):
    """
    Asigna el tema técnico (una de las 7 categorías) mediante mapeo semántico por palabras clave.
    Prioridad 1: coincidencias en título + texto + conceptos (conteo de matches, gana el máximo).
    Prioridad 2 (fallback): el tema bajo el cual se realizó la búsqueda original en la API,
    de forma que ningún registro quede sin clasificar dentro de los 7 temas requeridos.
    """
    combined = f'{titulo} {texto} {concepts_keywords}'.lower()
    scores = {}
    for category, keywords in CATEGORY_KEYWORDS.items():
        matches = sum(1 for kw in keywords if re.search(r'\b' + re.escape(kw) + r'\b', combined))
        if matches > 0:
            scores[category] = matches

    if scores:
        return max(scores, key=scores.get)
    return fallback_category

In [ ]:
# Construcción del esquema final (5 columnas) a partir de los registros crudos
schema_rows = []

for work in raw_records:
    titulo = work.get('title') or ''
    abstract = reconstruct_abstract(work.get('abstract_inverted_index'))
    concepts_keywords = get_concepts_keywords(work.get('concepts'))

    # texto = abstract + conceptos/temario secundario, unificado y limpio
    texto = f'{abstract} {concepts_keywords}'.strip()

    categoria = assign_category(titulo, abstract, concepts_keywords, work.get('_search_category'))

    schema_rows.append({
        'titulo': titulo,
        'texto': texto,
        'categoria': categoria,
        'autor': get_first_author(work.get('authorships')),
        # La columna tema se ha decidido mantener unificada en texto
        # 'tema': concepts_keywords,
        'tipo': normalize_tipo(work.get('type')),
    })

df_schema = pd.DataFrame(schema_rows)
print(f'Registros con esquema construido: {len(df_schema)}')
df_schema.sample(min(3, len(df_schema)))

#### Limpieza de la columna `texto`
Eliminamos etiquetas HTML/URLs residuales, valores nulos o vacíos y duplicados exactos, tal como exige el uso posterior en TF-IDF / Embeddings.

In [ ]:
def clean_html_urls(text):
    """Elimina etiquetas HTML y URLs de un texto."""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'http\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()


df_schema['titulo'] = df_schema['titulo'].apply(clean_html_urls)
df_schema['texto'] = df_schema['texto'].apply(clean_html_urls)
#df_schema['tema'] = df_schema['tema'].apply(clean_html_urls)

# Eliminar registros sin texto útil (abstract no disponible en OpenAlex por licenciamiento, etc.)
before = len(df_schema)
df_schema['texto'] = df_schema['texto'].replace('', np.nan)
df_schema = df_schema.dropna(subset=['texto']).reset_index(drop=True)
print(f'Registros sin texto eliminados: {before - len(df_schema)} (restantes: {len(df_schema)})')

# Eliminar duplicados exactos por título
before = len(df_schema)
df_schema = df_schema.drop_duplicates(subset='titulo').reset_index(drop=True)
print(f'Duplicados por título eliminados: {before - len(df_schema)} (restantes: {len(df_schema)})')

# Filtro de calidad mínima: al menos 100 palabras en 'texto' para ser útil en NLP
before = len(df_schema)
df_schema = df_schema[df_schema['texto'].str.split().str.len() >= 100].reset_index(drop=True)
print(f'Textos con menos de 100 palabras eliminados: {before - len(df_schema)} (restantes: {len(df_schema)})')

# Verificación final de nulos en 'texto'
assert df_schema['texto'].isna().sum() == 0, 'Aún existen valores nulos en texto'
assert (df_schema['texto'].str.strip() == '').sum() == 0, 'Aún existen textos vacíos'
print('\n Columna "texto" libre de nulos, HTML y URLs.')

## FASE 3: Segmentación y Balanceo por Umbrales

Distribución cruda por tema tras la limpieza, refuerzo automático y selección según las cuotas necesarias para completar 200 registros por categoría en el dataset unificado.

In [ ]:
REQUIRED_THEMES = list(TARGET_COUNTS_OPENALEX.keys())
MIN_THRESHOLD = 30

print('--- Distribución tras limpieza (previo a balanceo) ---')
print(df_schema['categoria'].value_counts())

#### Refuerzo de temas por debajo de la cuota
Si tras la limpieza algún tema no alcanza su cuota, se amplía la búsqueda en la API de OpenAlex con términos adicionales y sin recurrir a sobremuestreo.

In [ ]:
# Categorías que requieren una búsqueda adicional porque quedaron por debajo del umbral.
THEMES_TO_REINFORCE = [
    'Cloud',
    'Frontend',
]

# Términos alternativos de búsqueda temporales para cada categoría a reforzar.
# Se agregaron más variaciones a 'Cloud' para garantizar que traiga artículos suficientes.
FALLBACK_SEARCH_TERMS = {
    'Cloud': ['cloud computing', 'cloud infrastructure', 'aws', 'azure', 'serverless', 'cloud storage', 'distributed computing'],
    'Frontend': ['frontend', 'react', 'vue', 'user interface', 'web development']
}

# Umbral mínimo que debe alcanzar cada categoría antes del balanceo final.
REINFORCEMENT_THRESHOLD = 50

# Cantidad actual de registros por categoría después de la limpieza.
current_counts = df_schema['categoria'].value_counts().to_dict()

# Aquí se almacenarán únicamente los nuevos registros incorporados durante el refuerzo.
reinforced_rows = []

# Recorre únicamente las categorías que necesitan refuerzo.
for theme in THEMES_TO_REINFORCE:

    # Cantidad actual de registros disponibles para la categoría.
    count = current_counts.get(theme, 0)

    # Si por alguna razón la categoría ya alcanzó el mínimo,
    # no es necesario volver a consultar la API.
    if count >= REINFORCEMENT_THRESHOLD:
        print(f' "{theme}" ya cumple el umbral ({count} registros).')
        continue

    print(f' "{theme}" tiene {count} registros (< {REINFORCEMENT_THRESHOLD}). Ampliando búsqueda en la API...')

    # Realiza una nueva búsqueda utilizando términos alternativos
    # definidos específicamente para reforzar esta categoría.
    # Se aumentó raw_target y max_pages_per_term para dar margen a que pasen el filtro de 100 palabras y duplicados.
    extra_raw = fetch_category_raw(
        theme,
        FALLBACK_SEARCH_TERMS[theme],
        raw_target=100,
        max_pages_per_term=3
    )

    # Obtiene los IDs ya descargados para evitar incorporar
    # nuevamente artículos existentes.
    existing_ids = {work.get('id') for work in raw_records}

    # Conserva únicamente artículos completamente nuevos.
    extra_raw = [
        work
        for work in extra_raw
        if work.get('id') not in existing_ids
    ]

    # Convierte cada artículo nuevo al mismo esquema utilizado
    # por el DataFrame principal.
    for work in extra_raw:

        titulo = work.get('title') or ''

        # Reconstruye el abstract desde el índice invertido de OpenAlex.
        abstract = reconstruct_abstract(
            work.get('abstract_inverted_index')
        )

        # Obtiene los conceptos principales del artículo.
        concepts_keywords = get_concepts_keywords(
            work.get('concepts')
        )

        # Une abstract y conceptos para formar el texto que
        # posteriormente utilizará el modelo.
        texto = clean_html_urls(
            f'{abstract} {concepts_keywords}'.strip()
        )

        # Descarta registros con muy poco contenido textual.
        if len(texto.split()) < 100:
            continue

        reinforced_rows.append({
            'titulo': clean_html_urls(titulo),
            'texto': texto,
            'categoria': theme,
            'autor': get_first_author(work.get('authorships')),
            'tema': clean_html_urls(concepts_keywords),
            'tipo': normalize_tipo(work.get('type')),
        })

    # Agrega también los registros crudos a la colección principal
    # para mantener sincronizadas ambas estructuras de datos.
    raw_records.extend(extra_raw)

# Si se incorporaron nuevos registros,
# se agregan al DataFrame principal.
if reinforced_rows:

    df_reinforced = pd.DataFrame(reinforced_rows)

    df_schema = pd.concat(
        [df_schema, df_reinforced],
        ignore_index=True
    )

    # Elimina posibles duplicados basándose en el título.
    df_schema = (
        df_schema
        .drop_duplicates(subset='titulo')
        .reset_index(drop=True)
    )

    print(f'\n {len(reinforced_rows)} registros adicionales incorporados.')

else:

    print('\n No fue necesario incorporar registros adicionales.')

# Muestra la distribución final después del refuerzo.
print('\n--- Distribución tras refuerzo ---')
print(df_schema['categoria'].value_counts())

In [ ]:
# Refuerzo dinámico según las cuotas requeridas
FALLBACK_SEARCH_TERMS_QUOTAS = {
    'Backend': ['backend systems', 'server side architecture', 'REST API', 'microservices backend'],
    'Frontend': ['frontend framework', 'web user interface', 'responsive web application', 'single page web app'],
    'Data Science': ['applied machine learning', 'data mining', 'neural networks', 'artificial intelligence'],
    'Cloud': ['cloud services', 'serverless architecture', 'cloud native systems', 'distributed cloud'],
    'DevOps': ['DevOps automation', 'continuous delivery', 'container orchestration', 'infrastructure automation'],
    'Bases de Datos': ['relational database', 'NoSQL systems', 'database optimization', 'data warehouse'],
    'Mobile': ['Android development', 'iOS application', 'cross platform mobile', 'smartphone application'],
}

current_counts = df_schema['categoria'].value_counts().to_dict()
themes_to_reinforce = [
    theme
    for theme, target in CANDIDATE_TARGET_COUNTS_OPENALEX.items()
    if current_counts.get(theme, 0) < target
]
quota_reinforced_rows = []
existing_ids = {work.get('id') for work in raw_records}

for theme in themes_to_reinforce:
    count = current_counts.get(theme, 0)
    candidate_target = CANDIDATE_TARGET_COUNTS_OPENALEX[theme]
    needed = candidate_target - count
    print(f' "{theme}" necesita {needed} candidatos adicionales. Ampliando búsqueda...')

    extra_raw = fetch_category_raw(
        theme,
        FALLBACK_SEARCH_TERMS_QUOTAS[theme],
        raw_target=max(needed * 4, 100),
        max_pages_per_term=6,
    )
    extra_raw = [work for work in extra_raw if work.get('id') not in existing_ids]

    for work in extra_raw:
        titulo = work.get('title') or ''
        abstract = reconstruct_abstract(work.get('abstract_inverted_index'))
        concepts_keywords = get_concepts_keywords(work.get('concepts'))
        texto = clean_html_urls(f'{abstract} {concepts_keywords}'.strip())

        if len(texto.split()) < 100:
            continue

        quota_reinforced_rows.append({
            'titulo': clean_html_urls(titulo),
            'texto': texto,
            'categoria': theme,
            'autor': get_first_author(work.get('authorships')),
            'tipo': normalize_tipo(work.get('type')),
        })

    raw_records.extend(extra_raw)
    existing_ids.update(work.get('id') for work in extra_raw)

if quota_reinforced_rows:
    df_schema = pd.concat([df_schema, pd.DataFrame(quota_reinforced_rows)], ignore_index=True)
    df_schema = df_schema.drop_duplicates(subset=['titulo', 'texto']).reset_index(drop=True)

available_counts = df_schema['categoria'].value_counts().reindex(REQUIRED_THEMES, fill_value=0)
hard_shortages = {
    theme: TARGET_COUNTS_OPENALEX[theme] - int(available_counts[theme])
    for theme in REQUIRED_THEMES
    if available_counts[theme] < TARGET_COUNTS_OPENALEX[theme]
}
if hard_shortages:
    raise ValueError(f'OpenAlex no reunió suficientes registros únicos: {hard_shortages}')

# Preselección sin reemplazo; el buffer absorbe posibles fallos de traducción
balanced_groups = []
for theme in REQUIRED_THEMES:
    group = df_schema[df_schema['categoria'] == theme]
    candidate_n = min(len(group), CANDIDATE_TARGET_COUNTS_OPENALEX[theme])
    balanced_groups.append(group.sample(n=candidate_n, random_state=42, replace=False))

df_balanced = pd.concat(balanced_groups, ignore_index=True)
print('\n Candidatos OpenAlex preseleccionados por categoría:')
print(df_balanced['categoria'].value_counts().reindex(REQUIRED_THEMES))

## Traducción y procesamiento NLP

In [ ]:
tqdm.pandas()

translation_errors = []

def translate_to_spanish(text):
    try:
        return GoogleTranslator(source='en', target='es').translate(str(text)[:1500])
    except Exception as e:
        translation_errors.append(type(e).__name__)
        return ""

# Traduce título y texto principal al español
df_balanced['titulo_es'] = df_balanced['titulo'].progress_apply(translate_to_spanish)
df_balanced['texto_es'] = df_balanced['texto'].progress_apply(translate_to_spanish)

if translation_errors:
    print(f"⚠️ {len(translation_errors)} traducciones fallaron. Tipos de error: {Counter(translation_errors)}")

def clean_nlp(text):
    # Quita signos de puntuación y pasa a minúsculas
    text = re.sub(r'[^\w\sáéíóúñ]', ' ', str(text).lower())
    # Elimina stopwords en español y palabras muy cortas
    return ' '.join([word for word in text.split() if word not in spanish_stopwords and len(word) > 2])

df_balanced['texto_limpio'] = df_balanced['texto_es'].apply(clean_nlp)

# Elimina filas donde la traducción del texto falló
initial_count = len(df_balanced)
df_balanced = df_balanced[df_balanced['texto_es'].str.strip() != ''].reset_index(drop=True)
print(f"\nTraducciones fallidas eliminadas: {initial_count - len(df_balanced)} (Filas restantes: {len(df_balanced)})")

# Guarda un respaldo antes de consolidar el esquema final
df_balanced.to_csv(f'{CARPETA_PROCESADOS}/translation_backup_openalex.csv', index=False)
print("✅ Traducción y procesamiento NLP completados.")

# Reemplaza las columnas originales y descarta las auxiliares en la exportación final
df_balanced['titulo'] = df_balanced['titulo_es']
df_balanced['texto'] = df_balanced['texto_es']

### Selección final según las cuotas de OpenAlex

In [ ]:
available_after_translation = df_balanced['categoria'].value_counts().reindex(REQUIRED_THEMES, fill_value=0)
translation_shortages = {
    theme: TARGET_COUNTS_OPENALEX[theme] - int(available_after_translation[theme])
    for theme in REQUIRED_THEMES
    if available_after_translation[theme] < TARGET_COUNTS_OPENALEX[theme]
}

if translation_shortages:
    raise ValueError(f'Faltan registros traducidos de OpenAlex: {translation_shortages}')

df_balanced = pd.concat([
    df_balanced[df_balanced['categoria'] == theme].sample(
        n=TARGET_COUNTS_OPENALEX[theme], random_state=42, replace=False
    )
    for theme in REQUIRED_THEMES
]).reset_index(drop=True)

print('✅ Cuotas finales de OpenAlex completadas sin sobremuestreo.')
print(df_balanced['categoria'].value_counts().reindex(REQUIRED_THEMES))

### Tabla de validación final

### Validación e Integridad del Dataset Final

Verifica que el dataset balanceado (`df_balanced`) cumpla con los criterios de calidad mínimos necesarios antes de continuar con la etapa de entrenamiento o análisis:
1. **Verificación de volumen por tema:** Confirma que todas las categorías alcancen exactamente la cuota definida en `TARGET_COUNTS_OPENALEX`.
2. **Calidad del texto:** Garantiza que la columna `texto` esté libre de datos faltantes y que no contenga residuos de etiquetas HTML.

**Funcionamiento**
* **Conteo estructurado:** Recalcula la distribución de registros por categoría usando `.value_counts()` y asegura el orden oficial alineándolo con `REQUIRED_THEMES`.
* **Tabla de validación:** Genera un `DataFrame` con el objetivo y una columna booleana (`cumple_objetivo`) que compara los registros disponibles con su cuota exacta.
* **Manejo de alertas:** Muestra una notificación de éxito si todas las categorías aprueban la condición o desglosa los temas deficientes mediante un filtro sobre la tabla.
* **Aserciones de integridad (`assert`):**
  * `df_balanced['texto'].isna().sum() == 0`: Lanza un error si detecta valores nulos o vacíos.
  * `~df_balanced['texto'].str.contains('<[^>]+>')`: Comprueba mediante expresiones regulares que el texto no contenga etiquetas HTML remanentes.

In [ ]:
final_counts = df_balanced['categoria'].value_counts().reindex(REQUIRED_THEMES)

print('=== TABLA DE VALIDACIÓN: REGISTROS POR TEMA ===\n')
validation_table = final_counts.to_frame(name='registros')

# Realiza el conteo para satisfacer la carencia de los datasets coursera y mslearn  
validation_table['objetivo'] = pd.Series(TARGET_COUNTS_OPENALEX)
validation_table['cumple_objetivo'] = validation_table['registros'] == validation_table['objetivo']
display(validation_table)

if validation_table['cumple_objetivo'].all():
    print('\n Todas las categorías cumplen su cuota exacta de OpenAlex.')
else:
    faltantes = validation_table[~validation_table['cumple_objetivo']]
    print(f'\n Categorías fuera de cuota: {faltantes.index.tolist()}')

# Verificación de integridad de la columna texto en el dataset final
assert df_balanced['texto'].isna().sum() == 0
assert not df_balanced['texto'].str.contains('<[^>]+>', regex=True).any()
print(' "texto" verificado: sin nulos y sin etiquetas HTML.')

### Exportación del dataset final
`dataset_FINAL_openalex.csv` con el esquema oficial de 5 columnas: `titulo, texto, categoria, autor, tipo`.

In [ ]:
final_columns = ['titulo', 'texto', 'categoria', 'autor', 'tipo']
df_final = df_balanced[final_columns].reset_index(drop=True)

if df_final.columns.duplicated().any():
    print(' ERROR: columnas duplicadas en el dataset final.')
else:
    print('Sin columnas duplicadas — esquema limpio.')

output_path = f'{CARPETA_PROCESADOS}/dataset_FINAL_openalex.csv'
df_final.to_csv(output_path, index=False)
print(f'\n Pipeline completado. Dataset exportado en: {Path(output_path).relative_to(project_root)}')

print('\n=== MUESTRA DE AUDITORÍA (10 registros aleatorios) ===')
display(df_final.sample(min(10, len(df_final)), random_state=42))